# Pandas Avanzado — Notebook Demostrativo
Breve explicación y ejemplos ejecutables.

## 1. Indexado avanzado

### Explicación
Creamos un DataFrame de ejemplo con columnas categóricas y numéricas para usarlo en ejemplos de indexado. Esto nos permitirá practicar selección por etiquetas (`loc`) y por posición (`iloc`).


In [1]:
import pandas as pd
import numpy as np

df = pd.DataFrame({
    'Ciudad':['Madrid','Sevilla','Madrid','Valencia'],
    'Ventas':[100,200,150,130],
    'Unidades':[5,7,6,4]
})

print("Pandas version:", pd.__version__)
df

Pandas version: 2.3.3


,Ciudad,Ventas,Unidades
0,Madrid,100,5
1,Sevilla,200,7
2,Madrid,150,6
3,Valencia,130,4


### Explicación
Esta celda muestra la estructura completa del DataFrame para visualizar las filas y columnas antes de hacer indexaciones específicas.


In [2]:
df.loc[0, 'Ventas']

np.int64(100)

### Explicación
`loc` accede por etiqueta: aquí seleccionamos la fila con índice 0 y la columna `Ventas` por su nombre (útil cuando el índice no es numérico o cuando queremos claridad semántica).


In [3]:
df.iloc[:, 1:3]

,Ventas,Unidades
0,100,5
1,200,7
2,150,6
3,130,4


### Explicación
`iloc` usa posiciones numéricas: seleccionamos todas las filas y las columnas en posición 1 y 2 (rebanado exclusión del límite superior). Ideal para manipular por índice interno sin depender de nombres.


## 2. Filtros y expresiones

In [4]:
df[(df['Ventas']>120) & (df['Unidades']<6)]

,Ciudad,Ventas,Unidades
3,Valencia,130,4


### Explicación
Combinamos condiciones con `&`: selecciona filas donde `Ventas > 120` y simultáneamente `Unidades < 6`. Paréntesis son obligatorios para evaluar correctamente cada comparación antes del AND.


In [5]:
df.query('Ventas > 120')

,Ciudad,Ventas,Unidades
1,Sevilla,200,7
2,Madrid,150,6
3,Valencia,130,4


In [ ]:
df[(df['Ventas']>120) & (df['Unidades']<6)]
df.query('Ventas > 120 and Unidades < 6')

,Ciudad,Ventas,Unidades
3,Valencia,130,4


### Explicación
`query` permite escribir filtros como cadena con sintaxis tipo expresión: más legible en cadenas largas y evita muchos paréntesis. Aquí filtramos `Ventas > 120`.


## 3. GroupBy avanzado

In [7]:
df.groupby('Ciudad').agg({'Ventas':['sum','mean'],'Unidades':'sum'})

Ventas        Unidades
            sum   mean      sum
Ciudad                         
Madrid      250  125.0       11
Sevilla     200  200.0        7
Valencia    130  130.0        4

### Explicación
`groupby('Ciudad').agg({...})` aplica múltiples funciones a distintas columnas: `Ventas` recibe `sum` y `mean`; `Unidades` sólo `sum`. El resultado tiene MultiÍndice en columnas por combinar nombre y función.


## 4. Apply, Transform, Pipe

In [8]:
df.groupby('Ciudad')['Ventas'].transform('mean')

0    125.0
1    200.0
2    125.0
3    130.0
Name: Ventas, dtype: float64

### Explicación
`transform('mean')` calcula la media por grupo pero devuelve una Serie alineada al índice original (mismo tamaño que `df`), útil para crear columnas derivadas sin perder granularidad.


In [ ]:
df.groupby('Ciudad').apply(lambda g: g.nlargest(1,'Ventas'),include_groups=False)

Ventas  Unidades
Ciudad                      
Madrid   2     150         6
         0     100         5
Sevilla  1     200         7
Valencia 3     130         4

### Explicación
`apply` con `lambda` permite lógica personalizada por grupo. Aquí tomamos el registro con mayor `Ventas` en cada ciudad usando `nlargest(1,'Ventas')`.


## 5. Ventanas móviles

In [ ]:
s = pd.Series([1,3,5,7,9,11])
s.rolling(3).mean()

### Explicación
`rolling(3).mean()` calcula medias móviles de ventana tamaño 3 sobre la Serie. Las primeras posiciones insuficientes quedan como NaN.


## 6. Fechas

In [ ]:
fechas = pd.date_range('2024-01-01', periods=5, freq='D')
df2 = pd.DataFrame({'Fecha':fechas, 'Ventas':[5,6,7,8,9]})
df2.set_index('Fecha').resample('2D').sum()

### Explicación
Creamos un rango de fechas diario y luego `resample('2D').sum()` agrupa cada bloque de 2 días acumulando las ventas. Requiere índice de tipo fecha para funcionar.


## 7. Merge avanzado

In [ ]:
dfA = pd.DataFrame({'id':[1,2,3], 'x':[10,20,30]})
dfB = pd.DataFrame({'id':[2,3,4], 'y':[200,300,400]})
dfA.merge(dfB, on='id', how='outer')

### Explicación
`merge(... how='outer')` une ambos DataFrames conservando todas las claves (`id`) de A y B. Las filas sin correspondencia rellenan con NaN en las columnas faltantes.


## 8. Optimización

In [ ]:
df_optim = df.copy()
df_optim['Ciudad'] = df_optim['Ciudad'].astype('category')
df_optim.dtypes

### Explicación
Convertimos la columna categórica `Ciudad` a tipo `category` para reducir uso de memoria y acelerar operaciones que repiten valores (codificación interna de categorías).
